In [58]:

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import silhouette_score, adjusted_rand_score

from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [59]:
# =========================================================
# 0. 설정
# =========================================================
DATA_DIR = "../data/topics"     # .data 파일들이 있는 폴더 경로
OUTPUT_DIR = "../data/process"  # 결과물(csv, png) 저장 폴더
N_CLUSTERS_RANGE = range(2, 31)        # 엘보우 탐색 범위 (이 안에서 최적 K 자동 선택)
MAX_FEATURES = 3000                    # TF-IDF 어휘 크기
SVD_COMPONENTS = 100                   # TF-IDF -> 차원축소 후 군집화 (차원의 저주 방지)
RANDOM_STATE = 42

import platform

# 운영체제별 한글 폰트 자동 설정 (Windows: 맑은 고딕, Mac: AppleGothic, Linux: NanumGothic)
_system = platform.system()
if _system == "Windows":
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif _system == "Darwin":
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

In [60]:
# =========================================================
# 1. 파일 로드 + 기본 노이즈 정제
# =========================================================
def load_raw_data(data_dir: str) -> pd.DataFrame:
    """51개 {aspect}_{product}_txt.data 파일을 읽어 문장 단위 DataFrame으로 변환."""
    files = glob.glob(os.path.join(data_dir, "*.data"))
    if not files:
        raise FileNotFoundError(f"{data_dir} 에서 .data 파일을 찾지 못했습니다.")

    records = []
    for fpath in files:
        fname = os.path.basename(fpath).replace("_txt.data", "").replace(".data", "")
        parts = fname.split("_")
        aspect = parts[0]
        product = "_".join(parts[1:]) if len(parts) > 1 else "unknown"

        with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = basic_clean(line)
                if line:
                    records.append({"aspect": aspect, "product": product, "raw_text": line})

    df = pd.DataFrame(records)
    print(f"[로드 완료] 총 {len(df)}개 문장, {df['product'].nunique()}개 제품, {df['aspect'].nunique()}개 속성")
    return df


def basic_clean(text: str) -> str:
    """VADER 적용 전 최소한의 노이즈 정제 (원문의 대소문자/구두점은 보존)."""
    text = text.strip()
    text = text.replace("\r", "").replace("\n", " ")
    text = re.sub(r"\s*,\s*(?=\d)", "", text)   # "3, Cell" -> "3Cell" 같은 깨진 쉼표 정리
    text = re.sub(r"\s{2,}", " ", text)          # 중복 공백 정리
    text = text.strip(" .,")
    return text


In [61]:

# =========================================================
# 2. VADER 감성 점수 계산 (원문 기준)
# =========================================================
def add_sentiment_scores(df: pd.DataFrame) -> pd.DataFrame:
    sia = SentimentIntensityAnalyzer()
    df["sentiment"] = df["raw_text"].apply(lambda x: sia.polarity_scores(x)["compound"])

    # 검증: 극단 문장 몇 개 눈으로 확인
    print("\n[VADER 검증] 가장 긍정적인 문장 3개")
    print(df.nlargest(3, "sentiment")[["raw_text", "sentiment"]].to_string(index=False))
    print("\n[VADER 검증] 가장 부정적인 문장 3개")
    print(df.nsmallest(3, "sentiment")[["raw_text", "sentiment"]].to_string(index=False))

    return df, sia


In [62]:
# =========================================================
# 3. 감성어를 제외한 군집화용 정제
# =========================================================
def build_sentiment_word_set(sia: SentimentIntensityAnalyzer) -> set:
    """VADER lexicon에서 감성 강도가 있는 단어를 뽑아 stopword처럼 사용."""
    sentiment_words = {w.lower() for w, score in sia.lexicon.items() if abs(score) >= 1.0}
    return sentiment_words


def clean_for_cluster(text: str, stop_words: set, sentiment_words: set,
                       lemmatizer: WordNetLemmatizer) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(t) for t in tokens
        if t not in stop_words
        and t not in sentiment_words   # 감성어 제외 -> 군집이 '주제'만 담도록
        and len(t) > 2
    ]
    return " ".join(tokens)

In [63]:
# =========================================================
# 4. TF-IDF + SVD + (엘보우로 K 결정) + KMeans 군집분석
# =========================================================
def vectorize_and_reduce(df: pd.DataFrame, max_features: int, svd_components: int):
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        min_df=3,
        max_df=0.9,
        ngram_range=(1, 2),
    )
    tfidf_matrix = vectorizer.fit_transform(df["clean_text"])

    # 차원축소 (TF-IDF 수천 차원 -> 수십~백여 차원으로 줄여 KMeans 성능/속도 개선)
    svd = TruncatedSVD(n_components=svd_components, random_state=RANDOM_STATE)
    reduced_matrix = svd.fit_transform(tfidf_matrix)

    return vectorizer, svd, reduced_matrix


def find_optimal_k(reduced_matrix, k_range) -> tuple:
    """
    k_range 각각에 대해 KMeans inertia를 계산하고,
    inertia 곡선에서 첫점-끝점을 잇는 직선과 가장 멀리 떨어진 지점(꺾이는 지점)을
    최적 K로 자동 판단한다 (elbow method).
    """
    inertias = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        km.fit(reduced_matrix)
        inertias.append(km.inertia_)

    ks = np.array(list(k_range), dtype=float)
    inertias_arr = np.array(inertias, dtype=float)

    # 첫점, 끝점을 잇는 직선에서 각 점까지의 수직거리 계산 -> 최대 지점이 elbow
    x1, y1 = ks[0], inertias_arr[0]
    x2, y2 = ks[-1], inertias_arr[-1]
    line_vec = np.array([x2 - x1, y2 - y1])
    line_len = np.linalg.norm(line_vec)
    line_vec_norm = line_vec / line_len

    distances = []
    for x, y in zip(ks, inertias_arr):
        point_vec = np.array([x - x1, y - y1])
        proj_len = np.dot(point_vec, line_vec_norm)
        proj_point = proj_len * line_vec_norm
        distances.append(np.linalg.norm(point_vec - proj_point))

    optimal_idx = int(np.argmax(distances))
    optimal_k = int(ks[optimal_idx])

    # 엘보우 그래프 저장
    plt.figure(figsize=(8, 5))
    plt.plot(ks, inertias_arr, marker="o")
    plt.axvline(optimal_k, color="red", linestyle="--", label=f"선택된 K={optimal_k}")
    plt.title("엘보우 방법 - 클러스터 수(K)별 Inertia")
    plt.xlabel("클러스터 수 (K)")
    plt.ylabel("Inertia (군집 내 분산)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "elbow_method.png"), dpi=150)
    plt.close()

    print(f"\n[엘보우 방법] K={list(k_range)} 중 최적 K = {optimal_k} 로 결정됨")
    print(f"  (그래프 저장: {os.path.join(OUTPUT_DIR, 'elbow_method.png')})")

    return optimal_k, inertias_arr


def run_clustering(df: pd.DataFrame, reduced_matrix, n_clusters: int):
    kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10)
    cluster_labels = kmeans.fit_predict(reduced_matrix)

    cluster_cols = [f"cluster_{i}" for i in range(n_clusters)]
    # 군집 라벨을 원-핫 인코딩하여 회귀 피처로 사용 (hard assignment)
    cluster_dummies = pd.get_dummies(
        pd.Categorical(cluster_labels, categories=range(n_clusters)),
        prefix="cluster"
    )
    cluster_dummies.columns = cluster_cols
    cluster_dummies.index = df.index

    df = df.copy()
    df["cluster"] = [f"cluster_{c}" for c in cluster_labels]

    return kmeans, cluster_dummies, cluster_cols, df


def print_top_words_per_cluster(vectorizer: TfidfVectorizer, svd: TruncatedSVD,
                                 kmeans: KMeans, n_clusters: int, n_top=10):
    """군집 중심(centroid)을 SVD 역변환하여 원래 TF-IDF 공간 기준 상위 단어 추출."""
    feature_names = vectorizer.get_feature_names_out()
    # SVD 축소공간의 군집 중심 -> 원래 TF-IDF 차원으로 역변환(근사)
    original_space_centroids = svd.inverse_transform(kmeans.cluster_centers_)

    cluster_labels_dict = {}
    print("\n[군집별 대표 단어]")
    for idx in range(n_clusters):
        top_indices = original_space_centroids[idx].argsort()[::-1][:n_top]
        top_words = [feature_names[i] for i in top_indices]
        print(f"  cluster_{idx}: {', '.join(top_words)}")
        cluster_labels_dict[f"cluster_{idx}"] = ", ".join(top_words[:3])  # 상위 3단어로 임시 라벨
    return cluster_labels_dict


In [64]:

# =========================================================
# 5. 회귀 (Ridge) - 군집 -> 감성 해석
# =========================================================
def run_regression(df: pd.DataFrame, cluster_dummies: pd.DataFrame, cluster_cols: list):
    product_dummies = pd.get_dummies(df["product"], prefix="product")
    aspect_dummies = pd.get_dummies(df["aspect"], prefix="aspect")

    X = pd.concat([cluster_dummies, product_dummies, aspect_dummies], axis=1)
    y = df["sentiment"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )

    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    print(f"\n[회귀 결과] RMSE={rmse:.4f}, R2={r2:.4f}  (참고용 지표, 목적은 해석)")
    print(f"[회귀식 절편(intercept)] β0 = {model.intercept_:.4f}")

    # 군집 계수만 추출 (해석의 핵심)
    coef_series = pd.Series(model.coef_, index=X.columns)
    cluster_coef = coef_series[cluster_cols].sort_values()

    return model, cluster_coef

In [65]:
# =========================================================
# 5-b. 회귀 (OLS, statsmodels) - 통계적 유의성 검정용
# =========================================================
def run_regression_ols(df: pd.DataFrame, cluster_labels: dict):
    """
    Ridge는 정규화 회귀라 p-value가 없으므로, 통계적 유의성 검정을 위해
    statsmodels OLS로 별도 적합한다.

    - 각 범주형 변수(군집/제품/속성)에서 첫 범주를 기준범주로 제거(drop_first=True)
      하여 완전 다중공선성(더미변수 함정)을 우선 해소한다.
      (VIF를 통한 정식 다중공선성 점검은 별도 단계에서 다룬다.)
    - 이분산성 가능성을 감안해 강건표준오차(HC3)를 사용한다.
    """
    cluster_dummies = pd.get_dummies(df["cluster"], drop_first=True)  # 값 자체가 "cluster_0" 형태라 prefix 불필요
    product_dummies = pd.get_dummies(df["product"], prefix="product", drop_first=True)
    aspect_dummies = pd.get_dummies(df["aspect"], prefix="aspect", drop_first=True)

    X = pd.concat([cluster_dummies, product_dummies, aspect_dummies], axis=1).astype(float)
    X = sm.add_constant(X)
    y = df["sentiment"].astype(float)

    ols_model = sm.OLS(y, X).fit(cov_type="HC3")  # HC3: 이분산-강건표준오차

    print("\n" + "=" * 70)
    print("[OLS 회귀 결과] (강건표준오차 HC3 적용)")
    print("=" * 70)
    print(f"기준범주(reference) : cluster_0, 각 product/aspect의 알파벳순 첫 범주")
    print(f"R-squared (OLS)     : {ols_model.rsquared:.4f}")
    print(f"Adj. R-squared      : {ols_model.rsquared_adj:.4f}")
    print(f"F-statistic         : {ols_model.fvalue:.2f}  (p={ols_model.f_pvalue:.4g})")
    print(f"N (관측치 수)        : {int(ols_model.nobs)}")

    # 군집 항목만 추출해 보고서용 표로 정리
    cluster_param_names = [c for c in X.columns if c.startswith("cluster_")]
    ols_table = pd.DataFrame({
        "coef": ols_model.params[cluster_param_names],
        "std_err": ols_model.bse[cluster_param_names],
        "t": ols_model.tvalues[cluster_param_names],
        "p_value": ols_model.pvalues[cluster_param_names],
        "ci_lower": ols_model.conf_int().loc[cluster_param_names, 0],
        "ci_upper": ols_model.conf_int().loc[cluster_param_names, 1],
    })

    def _stars(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        elif p < 0.1:
            return "."
        return ""

    ols_table["sig"] = ols_table["p_value"].apply(_stars)
    ols_table["label"] = [cluster_labels.get(c, "") for c in ols_table.index]
    ols_table = ols_table.sort_values("coef", ascending=False)

    print("\n[군집별 OLS 회귀계수 (기준범주: cluster_0)]")
    print(ols_table[["coef", "std_err", "t", "p_value", "sig"]].round(4).to_string())
    print("\n(유의수준: *** p<0.001, ** p<0.01, * p<0.05, . p<0.1)")

    return ols_model, ols_table, X, y

In [66]:
# =========================================================
# 5-c. 다중공선성 점검 (VIF)
# =========================================================
def compute_vif(X: pd.DataFrame, label: str = "") -> pd.DataFrame:
    """
    OLS에 사용한 설계행렬(X, 상수항 포함)을 받아 예측변수별 VIF(분산팽창계수)를 계산한다.
    상수항(const)은 VIF 계산에서 제외한다.

    해석 기준(일반적 경험칙):
      VIF < 5   : 다중공선성 우려 낮음
      5 <= VIF < 10 : 다소 주의 필요
      VIF >= 10 : 심각한 다중공선성 의심
    """
    feature_cols = [c for c in X.columns if c != "const"]
    X_feat = X[feature_cols].values

    vif_values = [variance_inflation_factor(X_feat, i) for i in range(len(feature_cols))]
    vif_table = pd.DataFrame({"variable": feature_cols, "VIF": vif_values})
    vif_table = vif_table.sort_values("VIF", ascending=False).reset_index(drop=True)

    print("\n" + "=" * 70)
    print(f"[다중공선성 점검] VIF (분산팽창계수) - {label}")
    print("=" * 70)
    print(f"VIF >= 10 (심각) : {(vif_table['VIF'] >= 10).sum()}개 변수")
    print(f"5 <= VIF < 10 (주의) : {((vif_table['VIF'] >= 5) & (vif_table['VIF'] < 10)).sum()}개 변수")
    print(f"VIF < 5 (양호) : {(vif_table['VIF'] < 5).sum()}개 변수")

    print("\n[VIF 상위 10개 변수]")
    print(vif_table.head(10).round(3).to_string(index=False))

    cluster_vif = vif_table[vif_table["variable"].str.startswith("cluster_")]
    print("\n[군집(cluster) 변수만의 VIF]")
    print(cluster_vif.round(3).to_string(index=False))

    return vif_table

In [67]:
# =========================================================
# 5-d. VIF 기반 단계적 제거(stepwise elimination) + 재적합
# =========================================================
def reduce_multicollinearity(X: pd.DataFrame, protect_prefixes=("cluster_",),
                              vif_threshold: float = 10.0, max_iter: int = 100):
    """
    VIF가 threshold를 넘는 통제변수(product/aspect 더미)를 한 번에 하나씩 제거하며 재계산한다.
    - protect_prefixes로 시작하는 변수(기본: cluster_*)는 분석의 핵심 관심 변수이므로 제거 대상에서 제외한다.
    - VIF가 inf/NaN인 경우(완전 다중공선성) 최우선으로 제거한다.
    - 매 반복마다 통제변수 중 VIF가 가장 큰 변수 1개만 제거하고 다시 계산 -> 모두 threshold 미만이 될 때까지 반복.
    """
    X_reduced = X.copy()
    removed_log = []

    for _ in range(max_iter):
        feature_cols = [c for c in X_reduced.columns if c != "const"]
        candidate_cols = [c for c in feature_cols
                           if not any(c.startswith(p) for p in protect_prefixes)]
        if not candidate_cols:
            break

        X_feat = X_reduced[feature_cols].values
        vif_values = [variance_inflation_factor(X_feat, i) for i in range(len(feature_cols))]
        vif_series = pd.Series(vif_values, index=feature_cols)
        candidate_vif = vif_series[candidate_cols]

        # inf/NaN(완전 다중공선성)이 있으면 최우선으로 제거
        non_finite = candidate_vif[~np.isfinite(candidate_vif)]
        if len(non_finite) > 0:
            worst_col = non_finite.index[0]
            worst_vif = np.inf
        else:
            worst_vif = candidate_vif.max()
            if worst_vif < vif_threshold:
                break
            worst_col = candidate_vif.idxmax()

        X_reduced = X_reduced.drop(columns=[worst_col])
        removed_log.append((worst_col, worst_vif))

    print("\n" + "=" * 70)
    print(f"[VIF 기반 단계적 제거] threshold={vif_threshold}, 제거된 변수 {len(removed_log)}개")
    print("=" * 70)
    for col, v in removed_log:
        v_str = "inf" if not np.isfinite(v) else f"{v:.2f}"
        print(f"  제거: {col:45s} (제거 당시 VIF={v_str})")
    print(f"\n남은 변수 수: {X_reduced.shape[1]}개 (원래 {X.shape[1]}개, 상수항 포함)")

    return X_reduced, removed_log


def refit_ols(y: pd.Series, X: pd.DataFrame, cluster_labels: dict, label: str = ""):
    """정제된 설계행렬(X)로 OLS를 재적합하고, 군집 계수표를 다시 만든다."""
    model = sm.OLS(y, X).fit(cov_type="HC3")

    print("\n" + "=" * 70)
    print(f"[OLS 재적합 결과] {label} (강건표준오차 HC3 적용)")
    print("=" * 70)
    print(f"R-squared (OLS)     : {model.rsquared:.4f}")
    print(f"Adj. R-squared      : {model.rsquared_adj:.4f}")
    print(f"F-statistic         : {model.fvalue:.2f}  (p={model.f_pvalue:.4g})")
    print(f"N (관측치 수)        : {int(model.nobs)}")

    cluster_param_names = [c for c in X.columns if c.startswith("cluster_")]
    table = pd.DataFrame({
        "coef": model.params[cluster_param_names],
        "std_err": model.bse[cluster_param_names],
        "t": model.tvalues[cluster_param_names],
        "p_value": model.pvalues[cluster_param_names],
        "ci_lower": model.conf_int().loc[cluster_param_names, 0],
        "ci_upper": model.conf_int().loc[cluster_param_names, 1],
    })

    def _stars(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        elif p < 0.1:
            return "."
        return ""

    table["sig"] = table["p_value"].apply(_stars)
    table["label"] = [cluster_labels.get(c, "") for c in table.index]
    table = table.sort_values("coef", ascending=False)

    print(f"\n[군집별 OLS 회귀계수 - {label}]")
    print(table[["coef", "std_err", "t", "p_value", "sig"]].round(4).to_string())
    print("\n(유의수준: *** p<0.001, ** p<0.01, * p<0.05, . p<0.1)")

    return model, table

In [68]:
# =========================================================
# 5-e. 위계적 회귀 (군집 블록 F검정)
# =========================================================
def hierarchical_f_test(y: pd.Series, X_full: pd.DataFrame, cluster_prefix: str = "cluster_"):
    """
    통제변수(product/aspect)만 넣은 축소모형과, 여기에 군집 더미를 추가한
    완전모형을 비교하여 '군집'이라는 요인 블록 전체가 통계적으로 유의한
    설명력을 추가하는지 F검정(부분 F검정)한다.

    주의: 이 F검정은 등분산 가정을 전제로 하므로, 개별 계수 유의성 검정에
    쓴 HC3 강건표준오차 모형과 별개로 일반 OLS로 다시 적합한다.
    """
    control_cols = [c for c in X_full.columns if not c.startswith(cluster_prefix)]
    X_restricted = X_full[control_cols]

    model_full = sm.OLS(y, X_full).fit()             # 통제변수 + 군집
    model_restricted = sm.OLS(y, X_restricted).fit()  # 통제변수만

    f_stat, p_value, df_diff = model_full.compare_f_test(model_restricted)

    print("\n" + "=" * 70)
    print("[위계적 회귀] 군집 블록 F검정")
    print("=" * 70)
    print(f"축소모형(통제변수만)   R² = {model_restricted.rsquared:.4f}")
    print(f"완전모형(통제변수+군집) R² = {model_full.rsquared:.4f}")
    print(f"R² 증가분(ΔR²)         = {model_full.rsquared - model_restricted.rsquared:.4f}")
    print(f"제거된 자유도(군집 변수 개수) = {int(df_diff)}")
    print(f"F-statistic = {f_stat:.3f}")
    print(f"p-value     = {p_value:.4g}")

    if p_value < 0.05:
        print("\n=> 군집(cluster) 블록은 통계적으로 유의한 설명력을 추가한다 (p < 0.05).")
    else:
        print("\n=> 군집(cluster) 블록의 추가 설명력이 통계적으로 유의하지 않다 (p >= 0.05).")

    return model_full, model_restricted, f_stat, p_value, df_diff


In [69]:
# =========================================================
# 5-f. 군집 안정성 점검 (실루엣 점수 + ARI 재현성 검증)
# =========================================================
def check_cluster_stability(reduced_matrix, optimal_k: int, base_labels,
                             n_repeats: int = 10, base_seed: int = RANDOM_STATE):
    """
    1) 실루엣 점수: 채택한 K=optimal_k 군집이 얼마나 잘 분리되어 있는지 확인
       (범위 -1~1, 통상 0.5 이상이면 뚜렷, 0.25~0.5는 약하지만 실재하는 구조,
        0 근처는 군집 간 경계가 모호함을 의미. 텍스트 데이터는 낮게 나오는 경향이 있음)
    2) 서로 다른 random_state로 군집화를 반복하고, 최초 채택 결과와의
       ARI(Adjusted Rand Index, 1에 가까울수록 두 군집 결과가 일치)로 재현성을 확인
    """
    silhouette = silhouette_score(reduced_matrix, base_labels)

    print("\n" + "=" * 70)
    print("[군집 안정성] 실루엣 점수")
    print("=" * 70)
    print(f"K={optimal_k} 실루엣 점수 = {silhouette:.4f}")

    ari_scores = []
    seeds = [base_seed + i * 7 + 1 for i in range(n_repeats)]  # 기준 seed와 겹치지 않게 생성
    for seed in seeds:
        km = KMeans(n_clusters=optimal_k, random_state=seed, n_init=10)
        labels_i = km.fit_predict(reduced_matrix)
        ari = adjusted_rand_score(base_labels, labels_i)
        ari_scores.append(ari)

    ari_arr = np.array(ari_scores)

    print("\n" + "=" * 70)
    print(f"[군집 안정성] random_state {n_repeats}회 반복 재현성 (기준: 최초 채택 결과, seed={base_seed})")
    print("=" * 70)
    for seed, ari in zip(seeds, ari_scores):
        print(f"  seed={seed:4d} : ARI={ari:.4f}")
    print(f"\n평균 ARI = {ari_arr.mean():.4f}  표준편차 = {ari_arr.std():.4f}"
          f"  최소 = {ari_arr.min():.4f}  최대 = {ari_arr.max():.4f}")

    if ari_arr.mean() >= 0.8:
        print("=> 매우 안정적: 초기화(seed)가 달라져도 군집 구조가 거의 동일하게 재현됨")
    elif ari_arr.mean() >= 0.5:
        print("=> 중간 수준 안정성: 대체적인 구조는 유지되나 일부 군집 경계가 유동적")
    else:
        print("=> 낮은 안정성: seed에 따라 군집 구조가 상당히 달라짐 (해석에 주의 필요)")

    return silhouette, ari_arr

In [70]:
# =========================================================
# 6. 시각화
# =========================================================
def visualize(df, cluster_cols, cluster_labels, cluster_coef):
    df_full = df.reset_index(drop=True)  # df에는 이미 'cluster' 컬럼 포함되어 있음

    # (1) 군집별 회귀계수 (감성에 미치는 영향력, 방향)
    plt.figure(figsize=(9, 6))
    labels = [f"{c}\n({cluster_labels.get(c,'')})" for c in cluster_coef.index]
    colors = ["#d62728" if v < 0 else "#2ca02c" for v in cluster_coef.values]
    plt.barh(labels, cluster_coef.values, color=colors)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.title("군집별 감성 영향력 (Ridge 회귀계수)")
    plt.xlabel("계수 (감성 점수에 대한 영향)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "cluster_coefficients.png"), dpi=150)
    plt.close()

    # (2) 군집별 평균 감성 점수
    cluster_sentiment = df_full.groupby("cluster")["sentiment"].mean().reindex(cluster_cols)

    plt.figure(figsize=(9, 6))
    colors2 = ["#d62728" if v < 0 else "#2ca02c" for v in cluster_sentiment.values]
    plt.barh(cluster_sentiment.index, cluster_sentiment.values, color=colors2)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.title("군집별 평균 감성 점수")
    plt.xlabel("평균 VADER 감성 점수")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "cluster_mean_sentiment.png"), dpi=150)
    plt.close()

    # (3) 군집 x 제품 히트맵 (평균 감성)
    pivot = df_full.pivot_table(
        index="product", columns="cluster", values="sentiment", aggfunc="mean"
    ).reindex(columns=cluster_cols)

    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot, cmap="RdYlGn", center=0, annot=False, linewidths=0.3)
    plt.title("제품 x 군집 평균 감성 히트맵")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "product_cluster_heatmap.png"), dpi=150)
    plt.close()

    print(f"\n[시각화 저장 완료] {OUTPUT_DIR}/ 에 png 3개 저장됨")
    return df_full

In [71]:
# =========================================================
# 7. 체르노프 페이스 (Chernoff Face)
# =========================================================
def compute_cluster_stats(df_full: pd.DataFrame, cluster_cols: list, cluster_coef: pd.Series) -> pd.DataFrame:
    """군집별 통계치 집계: 평균감성, 표준편차, 문장수, 회귀계수."""
    stats = df_full.groupby("cluster")["sentiment"].agg(["mean", "std", "count"])
    stats = stats.reindex(cluster_cols)
    stats["coef"] = cluster_coef.reindex(cluster_cols)
    stats["std"] = stats["std"].fillna(0)  # 문장이 1개뿐인 군집 대비
    return stats


def _minmax(series: pd.Series) -> pd.Series:
    lo, hi = series.min(), series.max()
    if hi - lo == 0:
        return series * 0 + 0.5
    return (series - lo) / (hi - lo)


def draw_chernoff_face(ax, size_norm, eye_norm, eyebrow_norm, mouth_norm, coef_sign):
    """
    matplotlib 도형만으로 체르노프 페이스 하나를 그린다.
    size_norm, eye_norm : 0~1 (클수록 얼굴/눈이 큼)
    eyebrow_norm        : 0~1 (0=화난 눈썹 내림, 1=놀란 눈썹 올림)
    mouth_norm          : 0~1 (0=찡그림, 1=활짝 웃음)
    coef_sign           : 'pos' | 'neg' | 'neu' -> 얼굴색 결정
    """
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.axis("off")

    face_color = {"pos": "#c8f0c8", "neg": "#f5c6c6", "neu": "#e8e8e8"}[coef_sign]
    edge_color = {"pos": "#2ca02c", "neg": "#d62728", "neu": "#888888"}[coef_sign]

    # 얼굴
    radius = 0.28 + 0.14 * size_norm
    face = mpatches.Circle((0.5, 0.5), radius=radius, facecolor=face_color,
                            edgecolor=edge_color, linewidth=2.5, zorder=1)
    ax.add_patch(face)

    # 눈 (흰자 + 눈동자)
    eye_r = 0.035 + 0.045 * eye_norm
    for cx in (0.5 - radius * 0.42, 0.5 + radius * 0.42):
        cy = 0.5 + radius * 0.28
        white = mpatches.Ellipse((cx, cy), width=eye_r * 2, height=eye_r * 2.3,
                                  facecolor="white", edgecolor="black", linewidth=1, zorder=2)
        ax.add_patch(white)
        pupil = mpatches.Circle((cx, cy - eye_r * 0.1), radius=eye_r * 0.42,
                                 facecolor="black", zorder=3)
        ax.add_patch(pupil)

        # 눈썹: eyebrow_norm 0(화남, 눈쪽으로 처짐) ~ 1(놀람, 위로 올라감)
        tilt = (eyebrow_norm - 0.5) * 0.10   # -0.05 ~ +0.05
        inner_x = cx + (0.05 if cx < 0.5 else -0.05)
        by = cy + eye_r * 1.9
        ax.plot([inner_x, cx + (cx - inner_x) * -2],
                [by - tilt, by + tilt],
                color="black", linewidth=2.2, zorder=3)

    # 입: mouth_norm 0(찡그림) ~ 1(활짝 웃음)
    mouth_w = radius * 0.75
    mouth_h = 0.05 + 0.16 * abs(mouth_norm - 0.5) * 2
    mouth_cy = 0.5 - radius * 0.32
    if mouth_norm >= 0.5:
        theta1, theta2 = 200, 340   # 웃는 곡선(아래로 볼록)
    else:
        theta1, theta2 = 20, 160    # 찡그린 곡선(위로 볼록)
    mouth = mpatches.Arc((0.5, mouth_cy), width=mouth_w, height=mouth_h * 2,
                          angle=0, theta1=theta1, theta2=theta2,
                          color="black", linewidth=2.2, zorder=3)
    ax.add_patch(mouth)


def visualize_chernoff_faces(stats: pd.DataFrame, cluster_labels: dict, cluster_cols: list):
    """군집별 체르노프 페이스를 그리드로 그려 저장."""
    size_n = _minmax(stats["count"])
    eye_n = _minmax(stats["std"])
    eyebrow_n = _minmax(stats["coef"])
    mouth_n = _minmax(stats["mean"])

    n = len(cluster_cols)
    n_cols = 4
    n_rows = int(np.ceil(n / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4.6 * n_rows))
    axes = np.array(axes).reshape(-1)

    for i, c in enumerate(cluster_cols):
        ax = axes[i]
        coef_val = stats.loc[c, "coef"]
        coef_sign = "pos" if coef_val > 0.03 else ("neg" if coef_val < -0.03 else "neu")

        draw_chernoff_face(
            ax,
            size_norm=size_n[c],
            eye_norm=eye_n[c],
            eyebrow_norm=eyebrow_n[c],
            mouth_norm=mouth_n[c],
            coef_sign=coef_sign,
        )
        label = cluster_labels.get(c, "")
        ax.set_title(
            f"{c}\n({label})\n감성평균={stats.loc[c,'mean']:.2f}  계수={coef_val:+.2f}",
            fontsize=9,
        )

    for j in range(n, len(axes)):
        axes[j].axis("off")

    fig.suptitle(
        "군집별 체르노프 페이스\n"
        "(얼굴크기=문장수, 눈크기=감성편차, 눈썹=회귀계수, 입=평균감성, 색=긍정(초록)/부정(빨강))",
        fontsize=11,
    )
    plt.tight_layout(rect=[0, 0, 1, 0.90])
    out_path = os.path.join(OUTPUT_DIR, "cluster_chernoff_faces.png")
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"\n[체르노프 페이스 저장 완료] {out_path}")


In [72]:

# =========================================================
# 실행
# =========================================================
if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)  # data/process 폴더 없으면 생성

    # 1. 로드
    df = load_raw_data(DATA_DIR)

    # 2. VADER 감성 점수
    df, sia = add_sentiment_scores(df)

    # 3. 감성어 제외 정제
    stop_words = set(stopwords.words("english"))
    sentiment_words = build_sentiment_word_set(sia)
    lemmatizer = WordNetLemmatizer()

    df["clean_text"] = df["raw_text"].apply(
        lambda x: clean_for_cluster(x, stop_words, sentiment_words, lemmatizer)
    )
    df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

    # 4. 벡터화 + 차원축소 -> 엘보우로 최적 K 탐색 -> 군집화
    vectorizer, svd, reduced_matrix = vectorize_and_reduce(df, MAX_FEATURES, SVD_COMPONENTS)
    optimal_k, inertias = find_optimal_k(reduced_matrix, N_CLUSTERS_RANGE)
    kmeans, cluster_dummies, cluster_cols, df = run_clustering(df, reduced_matrix, optimal_k)
    cluster_labels = print_top_words_per_cluster(vectorizer, svd, kmeans, optimal_k)

    # 5. 회귀 (해석용, Ridge)
    model, cluster_coef = run_regression(df, cluster_dummies, cluster_cols)

    print("\n[감성을 가장 높이는 군집 TOP3]")
    print(cluster_coef.sort_values(ascending=False).head(3))
    print("\n[감성을 가장 낮추는 군집 TOP3]")
    print(cluster_coef.sort_values().head(3))

    # 5-b. 회귀 (통계적 유의성 검정용, OLS - 1차: 전체 변수)
    ols_model, ols_table, X_ols, y_ols = run_regression_ols(df, cluster_labels)
    ols_table_path = os.path.join(OUTPUT_DIR, "cluster_ols_regression_table.csv")
    ols_table.to_csv(ols_table_path, encoding="utf-8-sig")
    print(f"\n[OLS 회귀표 저장 완료] {ols_table_path}")

    # 5-c. 다중공선성 점검 (VIF, 1차)
    vif_table = compute_vif(X_ols, label="1차(전체 변수)")
    vif_table_path = os.path.join(OUTPUT_DIR, "cluster_vif_table.csv")
    vif_table.to_csv(vif_table_path, index=False, encoding="utf-8-sig")
    print(f"\n[VIF 표 저장 완료] {vif_table_path}")

    # 5-d. VIF 기준 문제 통제변수 단계적 제거 -> 재적합 (군집 변수는 항상 보존)
    X_reduced, removed_log = reduce_multicollinearity(X_ols, protect_prefixes=("cluster_",),
                                                        vif_threshold=10.0)
    ols_model_refined, ols_table_refined = refit_ols(y_ols, X_reduced, cluster_labels,
                                                       label="VIF 정제 후")
    ols_table_refined_path = os.path.join(OUTPUT_DIR, "cluster_ols_regression_table_refined.csv")
    ols_table_refined.to_csv(ols_table_refined_path, encoding="utf-8-sig")
    print(f"\n[정제 후 OLS 회귀표 저장 완료] {ols_table_refined_path}")

    # 정제 후 VIF도 다시 확인 (군집 변수 포함 전체가 기준치 아래로 내려왔는지 검증)
    vif_table_refined = compute_vif(X_reduced, label="정제 후")
    vif_table_refined_path = os.path.join(OUTPUT_DIR, "cluster_vif_table_refined.csv")
    vif_table_refined.to_csv(vif_table_refined_path, index=False, encoding="utf-8-sig")
    print(f"\n[정제 후 VIF 표 저장 완료] {vif_table_refined_path}")

    # 5-e. 위계적 회귀 (군집 블록이 통계적으로 유의한 설명력을 더하는지 F검정)
    model_full, model_restricted, f_stat, f_pvalue, df_diff = hierarchical_f_test(y_ols, X_reduced)

    # 5-f. 군집 안정성 점검 (실루엣 점수 + ARI 재현성)
    silhouette, ari_scores = check_cluster_stability(
        reduced_matrix, optimal_k, kmeans.labels_, n_repeats=10, base_seed=RANDOM_STATE
    )

    # 6. 시각화
    df_full = visualize(df, cluster_cols, cluster_labels, cluster_coef)

    # 7. 체르노프 페이스
    cluster_stats = compute_cluster_stats(df_full, cluster_cols, cluster_coef)
    visualize_chernoff_faces(cluster_stats, cluster_labels, cluster_cols)

    # 결과 저장
    result_path = os.path.join(OUTPUT_DIR, "cluster_sentiment_result.csv")
    df_full.to_csv(result_path, index=False, encoding="utf-8-sig")
    print(f"\n[전체 완료] 결과 CSV 저장: {result_path}")

[로드 완료] 총 7086개 문장, 13개 제품, 36개 속성

[VADER 검증] 가장 긍정적인 문장 3개
                                                                                                                                                                                                                                                                                                                                                                                                                                                                      raw_text  sentiment
my first fill up was 26 mpg mixed city and hwy I only expect it to get better, steering is tight and precise, only complaints are road noise is more than i like but its livable, rain or just dew pours in right on top of the power window controls when the window is cracked, but window guards have fixed that, its a fun car to drive, and for what it is, its comfortable, controls are great, easy to reach, I'm looking forward to a lot of great miles with this car   

c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 58, but rank is 56
  warnings.warn('covariance of constraints does not have full '
c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)



[다중공선성 점검] VIF (분산팽창계수) - 1차(전체 변수)
VIF >= 10 (심각) : 16개 변수
5 <= VIF < 10 (주의) : 15개 변수
VIF < 5 (양호) : 27개 변수

[VIF 상위 10개 변수]
                             variable          VIF
product_mileage_toyota_camry_2007.txt          inf
                           aspect_gas          inf
                          aspect_size          inf
      product_asus_netbook_1005ha.txt 4.195120e+09
    product_bestwestern_hotel_sfo.txt 2.873700e+01
        product_honda_accord_2008.txt 2.711800e+01
        product_toyota_camry_2007.txt 2.610300e+01
       product_holiday_inn_london.txt 2.495500e+01
                      aspect_location 1.676700e+01
                       aspect_comfort 1.548500e+01

[군집(cluster) 변수만의 VIF]
  variable   VIF
 cluster_2 6.483
 cluster_7 5.343
 cluster_6 5.034
cluster_11 4.772
 cluster_9 4.045
 cluster_1 3.995
 cluster_4 3.471
cluster_10 2.433
 cluster_5 2.329
 cluster_3 1.876
 cluster_8 1.636

[VIF 표 저장 완료] ../data/process\cluster_vif_table.csv


c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)



[VIF 기반 단계적 제거] threshold=10.0, 제거된 변수 4개
  제거: product_mileage_toyota_camry_2007.txt         (제거 당시 VIF=inf)
  제거: product_asus_netbook_1005ha.txt               (제거 당시 VIF=inf)
  제거: product_bestwestern_hotel_sfo.txt             (제거 당시 VIF=28.74)
  제거: product_honda_accord_2008.txt                 (제거 당시 VIF=27.12)

남은 변수 수: 55개 (원래 59개, 상수항 포함)

[OLS 재적합 결과] VIF 정제 후 (강건표준오차 HC3 적용)
R-squared (OLS)     : 0.1416
Adj. R-squared      : 0.1350
F-statistic         : 27.79  (p=4.404e-250)
N (관측치 수)        : 7084

[군집별 OLS 회귀계수 - VIF 정제 후]
              coef  std_err       t  p_value  sig
cluster_8   0.3268   0.0447  7.3169   0.0000  ***
cluster_11  0.3179   0.0378  8.4194   0.0000  ***
cluster_3   0.2346   0.0316  7.4286   0.0000  ***
cluster_9   0.1385   0.0359  3.8614   0.0001  ***
cluster_7   0.1367   0.0364  3.7503   0.0002  ***
cluster_2   0.1207   0.0628  1.9217   0.0546    .
cluster_6   0.0461   0.0293  1.5748   0.1153     
cluster_5   0.0391   0.0581  0.6728   0.5011     
cluster_